In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# I import all libraries and tools used in the project

# Let's start the initial data audit
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# I display the training data for a quick visual inspection
train

# I check the shape of the training data
train.shape

# I inspect the frequency of complete rows as an additional audit check
train.value_counts()

# I count all missing values in the training data
train.isna().sum().sum()

# I inspect the data types of the training columns
train.dtypes

# I check the class distribution of the target variable
train['label'].value_counts().sort_index()

# I display the test data for a quick visual inspection
test

# I check the shape of the test data
test.shape

# I inspect the data types of the test columns
test.dtypes

# I count all missing values in the test data
test.isna().sum().sum()

# I inspect the frequency of complete test rows as an additional audit check
test.value_counts()

# Data Audit summary: I see the training set contains 42,000 rows and 785 columns
# Each row contains the label corresponding to the handwritten digit and 784 pixel columns from pixel0 to pixel783
# The test set has the same pixel structure but does not contain the label column
# All columns are int64, and I confirm there are no missing values
# The label distribution is close to uniform. Class 5 occurs least often, while 1, 7, and 3 are among the most frequent classes

# I get the 784 pixels of the 42,000th image and reshape them into a 28×28 matrix
matrix = train.drop(columns='label').iloc[41999].to_numpy().reshape(28, 28)

# I get the correct label of this image
label = train['label'].iloc[41999]

# I display the matrix and label
print(matrix)
print(label)

# I display the image in grayscale
plt.imshow(matrix, cmap='gray')

# I show the correct digit in the title
plt.title(f'Label: {label}')

# I remove the coordinate axes
plt.axis('off')

# I display the image
plt.show()

# Let's start the EDA — I want to understand how images from different classes look and whether they have visual patterns

fig, axes = plt.subplots(2, 5, figsize=(12, 6))  # I create a grid for 10 images

for i, ax in enumerate(axes.flat):
    # I find the first row where the label equals the current digit i
    row = train[train['label'] == i].drop(columns='label').iloc[0]
    # I reshape the 784 pixels into a 28×28 matrix
    matrix = row.to_numpy().reshape(28, 28)
    
    # I display the image
    ax.imshow(matrix, cmap='gray')

    # I show the corresponding digit
    ax.set_title(f'Label: {i}')

    # I remove the coordinate axes
    ax.axis('off')

plt.tight_layout()
plt.show()

# I see that images from different digits contain characteristic spatial elements, but some elements are shared by several classes
# Therefore, I conclude that classification depends on a combination of pixels rather than individual pixels

# I keep only the pixel columns and exclude the label column
pixels = train.drop(columns='label')

# I identify which pixel values are equal to zero
zero_pixels = pixels == 0

# I count the number of zero-valued pixels
zero_count = zero_pixels.sum().sum()

# I count the total number of pixels
total_count = pixels.size

# I calculate the proportion of zero-valued pixels
zero_ratio = zero_count / total_count

print(zero_ratio)

# I notice that most pixel features for a particular image are zero, while a relatively small part of the image carries information about the digit

# I get all pixel columns without the label column
pixels = train.drop(columns='label')

# I keep only non-zero pixel values
non_zero_pixels = pixels[pixels > 0]

# I remove missing values created by replacing zeros
non_zero_pixels = non_zero_pixels.stack()

# I plot the distribution of non-zero pixel values
plt.hist(non_zero_pixels, bins=50)

# I add the x-axis label
plt.xlabel('Pixel Value')

# I add the y-axis label
plt.ylabel('Number of Pixels')

# I add the plot title
plt.title('Distribution of Non-Zero Pixel Values')

# I display the plot
plt.show()

# I see that most non-zero pixels have high intensity, roughly above 250
# In summary, I conclude that most useful information about digit shape is contained in the location of bright pixels rather than only in their exact intensity

fig, axes = plt.subplots(3, 3, figsize=(6, 6))  # I create a grid for 9 images
# I filter all rows containing the digit 3 and exclude the label column
all_threes = train[train['label'] == 3].drop(columns='label')

for i, ax in enumerate(axes.flat):
    row = all_threes.iloc[i]
    # I reshape the 784 pixels into a 28×28 matrix
    matrix = row.to_numpy().reshape(28, 28)
    
    # I display the image
    ax.imshow(matrix, cmap='gray')

    # I show the corresponding digit
    ax.set_title(f'Label: {i}')

    # I remove the coordinate axes
    ax.axis('off')

plt.tight_layout()
plt.show()

# I notice that Digit 3 keeps a similar shape across samples,
# occupies approximately the same area, and often contains bright pixels close to the maximum intensity

# I select only rows where the correct digit is 3
threes = train[train['label'] == 3]

# I remove the label column, leaving 784 pixel features
threes_pixels = threes.drop(columns='label')

# I calculate the mean value of each pixel across all images of digit 3
mean_three = threes_pixels.mean(axis=0)

# I reshape the 784 mean values back into a 28×28 matrix
mean_three_matrix = mean_three.to_numpy().reshape(28, 28)

# I display the mean image of digit 3
plt.imshow(mean_three_matrix, cmap='gray')

# I add the title
plt.title('Mean Image of Digit 3')

# I remove the coordinate axes
plt.axis('off')

# I display the image
plt.show()

# I see that images from the same class share a common structure despite differences in the exact handwriting

fig, axes = plt.subplots(2, 5, figsize=(12, 6))  # I create a grid for 10 class-average images

for i, ax in enumerate(axes.flat):
    # I select only rows where the correct digit equals i
    number = train[train['label'] == i]

    # I remove the label column, leaving 784 pixel features
    number_pixels = number.drop(columns='label')

    # I calculate the mean value of each pixel across all images of digit i
    mean_number = number_pixels.mean(axis=0)

    # I reshape the 784 mean values back into a 28×28 matrix
    mean_number_matrix = mean_number.to_numpy().reshape(28, 28)

    # I display the image
    ax.imshow(mean_number_matrix, cmap='gray')

    # I show the corresponding digit
    ax.set_title(f'Label: {i}')

    # I remove the coordinate axes
    ax.axis('off')

plt.tight_layout()
plt.show()

'''
I see that each class has a common overall shape, but the position of individual strokes can vary noticeably between images
The greater the variation in stroke position, the stronger the blur becomes in the average image
'''

# My hypothesis: if pixel location and intensity contain sufficiently stable patterns for different digits,
# Logistic Regression should be able to separate the 10 classes reasonably well without complex non-linear modeling

x = train.drop(columns='label')  # I prepare the feature matrix
y = train['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

liner_model_e1 = LogisticRegression(random_state=42)
# I fit the model on the training split
liner_model_e1.fit(x_train, y_train)

# I generate predictions on the validation split
y_pred = liner_model_e1.predict(x_test)

# I calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# I print the result
print(accuracy)  # I check the result without scaling

'''
On this run, I see the model achieved accuracy of 0.909,
but training ended with a convergence warning, so I need to correct and recheck the baseline
'''

liner_model_e2 = LogisticRegression(random_state=42, max_iter=300)
# I fit the model on the training split
liner_model_e2.fit(x_train, y_train)

# I generate predictions on the validation split
y_pred_e2 = liner_model_e2.predict(x_test)

# I calculate accuracy
accuracy_e2 = accuracy_score(y_test, y_pred_e2)

# I print the result
print(accuracy_e2)  # I check the result after increasing max_iter

preprocessor = StandardScaler()

liner_model_e3 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=300))
])

liner_model_e3.fit(x_train, y_train)

y_pred_e3 = liner_model_e3.predict(x_test)
accuracy_e3 = accuracy_score(y_test, y_pred_e3)
print(accuracy_e3)  # I expect no convergence warning now

liner_model_e4 = Pipeline([("classifier", LogisticRegression(random_state=42, max_iter=300))])

# I fit the unscaled Logistic Regression pipeline on the training split
liner_model_e4.fit(x_train, y_train)

# I generate predictions on the validation split
y_pred_e4 = liner_model_e4.predict(x_test)

# I calculate validation accuracy
accuracy_e4 = accuracy_score(y_test, y_pred_e4)

# I print the validation accuracy
print(accuracy_e4)  # I check the result without scaling

'''
Experiment: StandardScaler + Logistic Regression
Change: I added StandardScaler and increased max_iter to 300
Result: accuracy = 0.9026; the convergence warning disappeared
Conclusion: I see that scaling helped the model converge,
but it did not improve validation accuracy compared with the incomplete unscaled run
'''

# I evaluate the pipeline using 5-fold cross-validation
cv_scores_e3 = cross_val_score(
    liner_model_e3,
    x,
    y,
    cv=5,
    scoring='accuracy'
)

# I print the accuracy for each fold
print(cv_scores_e3)

# I print the mean accuracy
print(cv_scores_e3.mean())

'''
My hypothesis:
Random Forest will capture non-linear relationships between pixels and achieve higher CV accuracy than Logistic Regression
'''

preprocessor = StandardScaler()

liner_model_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_estimators=100))
])

# I fit the Random Forest pipeline on the training split
liner_model_rf.fit(x_train, y_train)

# I generate predictions on the validation split
y_pred_rf = liner_model_rf.predict(x_test)

# I calculate validation accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

# I print the validation accuracy
print(accuracy_rf)

# I evaluate the Random Forest pipeline using 5-fold cross-validation
cv_scores_rf = cross_val_score(
    liner_model_rf,
    x,
    y,
    cv=5,
    scoring='accuracy'
)

# I print the accuracy for each fold
print(cv_scores_rf)

# I print the mean accuracy
print(cv_scores_rf.mean())

'''
Conclusion:
In 5-fold CV, I see Random Forest achieved mean accuracy of about 96.39%,
while Logistic Regression achieved about 90.60%. On these data, Random Forest performed noticeably better for digit classification,
so I moved to Random Forest hyperparameter tuning
'''

param_dist = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__max_features": ["sqrt", "log2", None]
}

rf_model_tun = Pipeline([
    ("classifier", RandomForestClassifier(random_state=42))
])

random_search_rf = RandomizedSearchCV(
    estimator=rf_model_tun,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

# I run the hyperparameter search on the training split
random_search_rf.fit(x_train, y_train)

# I store the best pipeline found by the search
best_rf = random_search_rf.best_estimator_

# I generate predictions from the tuned model on the validation split
y_pred_tuned = best_rf.predict(x_test)

# I calculate validation accuracy for the tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

# I print the tuned validation accuracy
print(accuracy_tuned)

# I print the best hyperparameter combination found by the search
print(random_search_rf.best_params_)

# I print the best cross-validation score from RandomizedSearchCV
print(random_search_rf.best_score_)

'''
I use RandomizedSearchCV to test the hypothesis:
changing Random Forest hyperparameters can improve accuracy
relative to the baseline result of about 0.961
I use cv=5 because it provides a more reliable estimate, although it increases computation time
I limit n_iter to 20 to keep the search computationally manageable
'''

cv_scores_rf_tun = cross_val_score(
    best_rf,
    x,
    y,
    cv=5,
    scoring='accuracy'
)

# I print the accuracy for each fold
print(cv_scores_rf_tun)

# I print the mean accuracy
print(cv_scores_rf_tun.mean())

# I notice the tuned hyperparameter combination did not produce a meaningful improvement

'''
My hypothesis: Random Forest will more often confuse 1 with 4 and 7 with 9
because the average images of these classes contain blurred stroke regions
that indicate variation in their shape and position
I will build a confusion matrix to test this hypothesis
'''

# I build the confusion matrix from the true and predicted classes,
# to test the hypothesis that 1 and 4, as well as 7 and 9,
# may be confused more often than other classes
cm = confusion_matrix(y_test, y_pred_tuned)
# I visualize the confusion matrix as an image,
# to compare the number of errors between classes
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar()

# I add the numeric count to each cell
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        plt.text(j, i, str(cm[i, j]),
                ha="center", va="center",
                color=color, fontsize=14)

# I add axis labels
ticks = np.arange(10)
plt.xticks(ticks, ticks)
plt.yticks(ticks, ticks)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')

plt.tight_layout()
plt.show()


'''
Error Analysis conclusion

The confusion matrix highlighted several noticeable classification errors for me:
1. **`1 → 6`** — I see some handwritten versions of `6` have a weak or incomplete upper contour,
   which can make individual images visually similar to `1`
2. **`4 → 9`** — I notice some handwritten versions of these digits have similar arrangements of the main strokes, which can make them harder to distinguish
3. **`7 ↔ 9`** — I see some images show similarity in the upper part of the digits and in the direction of the main strokes

Overall, my error analysis suggests that some Random Forest errors may be related to handwriting variation
and visual similarity between particular digit variants
'''

# I test the hypothesis that errors in 4 → 9 are related to visual similarity between these digits
# I inspect real images
# that Random Forest classified as 9

# I create a boolean mask: True means
# the true digit is 4 and the model predicted 9
error_mask = (y_test == 4) & (y_pred_tuned == 9)

# I get the indices of the misclassified images
error_indices = y_test.index[error_mask]

# I keep the first 5 errors
# to inspect them manually without overcrowding the plot
error_indices = error_indices[:5]

# I retrieve the corresponding images from the original train DataFrame,
# remove the label column, and reshape the 784 pixels back into 28×28 images
pics = (train.loc[error_indices].drop(columns="label").to_numpy().reshape(-1, 28, 28))

# I create a figure for five misclassified images
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

# I plot each image and its corresponding axes object
for i, ax in enumerate(axes):
    ax.imshow(pics[i], cmap="gray")
    ax.set_title("True: 4 | Pred: 9")
    ax.axis("off")

plt.tight_layout()
plt.show()

'''
Result: among the 5 inspected `4 → 9` errors, I see 3 images showed noticeable visual similarity to `9`
This partially supports my hypothesis that
some Random Forest errors are related to handwriting variation

However, the number of errors for the individual pairs is small:
none of the investigated pairs exceeds 20 errors
Therefore, this analysis does not provide enough evidence for separate Feature Engineering
aimed specifically at these errors
'''

'''
Model selection conclusion

After RandomizedSearchCV, I see the tuned Random Forest did not show a meaningful accuracy improvement over the baseline model
Therefore, further unguided hyperparameter search is not sufficiently justified

Additional Error Analysis showed me that a noticeable part of the errors is related to visual variation in handwriting
For the most noticeable error pairs, such as `1 → 6`, `4 → 9`, and `7 ↔ 9`, some inspected examples had visual similarity to the predicted class
However, the number of errors in these pairs was relatively small,
so I did not find sufficient grounds for dedicated Feature Engineering aimed specifically at these cases
For the final stage, I selected the Random Forest with the hyperparameters found during tuning
The next step is to fit the selected model on the full labeled dataset and generate predictions for the Kaggle test data
'''



In [86]:
# I import the libraries required for the final training and submission
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# I load the labeled training dataset and the unlabeled Kaggle test dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


# I separate the target variable from the pixel features
# I use all 784 pixel columns as features because the previous experiments
# showed that the Random Forest can achieve high accuracy using the original pixel representation
x = train.drop(columns="label")
y = train["label"]


# I create a validation split with the same parameters used during the previous experiments
# I keep stratification so that all ten digit classes remain proportionally represented
x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# I create the final tuned Random Forest using the best hyperparameters found during tuning
# I use these parameters because the tuning experiment did not produce a substantial improvement,
# but I keep this configuration as the final tuned version of the model
final_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)


# I train the final tuned model on the training part of the validation split
final_rf.fit(x_train, y_train)


# I generate predictions for the validation data to obtain a final local accuracy estimate
y_valid_pred = final_rf.predict(x_valid)


# I calculate the validation accuracy before fitting the final model on all labeled data
final_validation_accuracy = accuracy_score(y_valid, y_valid_pred)


# I refit the selected model on the complete labeled training dataset
# I use all available labeled data for the final Kaggle prediction after completing model selection
final_rf.fit(x, y)


# I generate predictions for the Kaggle test dataset
test_predictions = final_rf.predict(test)


# I load the Kaggle submission template to preserve its required structure
sample_submission = pd.read_csv("sample_submission.csv")


# I verify that the submission template contains the same number of rows as the test dataset
if len(sample_submission) != len(test_predictions):
    raise ValueError("The submission template and test dataset contain different numbers of rows")


# I verify that the submission template contains the expected target column
if "Label" not in sample_submission.columns:
    raise ValueError("The submission template does not contain a 'Label' column")


# I insert the final predictions into the submission template
sample_submission["Label"] = test_predictions


# I save the final predictions in Kaggle submission format
submission_path = "submission.csv"
sample_submission.to_csv(submission_path, index=False)


# I store the final values needed by the last result cell
final_submission_rows = len(sample_submission)
final_test_rows = len(test)
final_submission_ready = True

print(f"Final validation accuracy: {final_validation_accuracy:.5f}")
print(f"Submission rows: {final_submission_rows}")
print(f"Test rows: {final_test_rows}")
print(f"Submission ready: {final_submission_ready}")

Final validation accuracy: 0.96095
Submission rows: 28000
Test rows: 28000
Submission ready: True
